# Final math-simplification comparison: before vs. \(\lambda=1\), unbounded \(\alpha\)

This notebook performs the **paired final comparison** between the previous full benchmark and the simplified formulation:

- **Before:** bounded nonlinear identifier with \(\lambda=0.70\).
- **After:** unbounded \(\alpha(z)=\mathrm{softplus}(f_\phi(z))/\log 2\) with \(\lambda=1\).
- **Regimes:** single-shot and abundant-data.

The comparison uses the same environment and learning-seed identities. It validates benchmark equivalence before reporting any performance deltas. All reported `new_minus_old` values are positive when the simplified formulation achieved a higher final mean opinion (the target is 1.0).

The notebook can be run interactively or with `run_ipynb_cells.py`.


In [1]:
from __future__ import annotations

import json
import math
import os
from pathlib import Path
from typing import Any, Iterable

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import t as student_t, ttest_1samp

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 300)
pd.set_option("display.max_rows", 200)


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "opinion_dynamics").exists():
            return candidate
    raise RuntimeError("Could not find repository root containing opinion_dynamics/.")


REPO_ROOT = find_repo_root()
EXPERIMENTS_DIR = REPO_ROOT / "opinion_dynamics" / "experiments"
RESULTS_ROOT = EXPERIMENTS_DIR / "results"

# Optional manual overrides. Leave as None for automatic discovery.
OVERRIDES: dict[str, str | None] = {
    "single_old": None,
    "single_new": None,
    "abundant_old": None,
    "abundant_new": None,
}

EXPECTED_DIR_NAMES = {
    "single_old": "experiment_2026_08_17_broad_all_dynamics_multitopology_multiseed",
    "single_new": "experiment_2026_09_17_broad_all_dynamics_multitopology_multiseed_unbounded_lambda1",
    "abundant_old": "experiment_2026_08_24_abundant_all_dynamics_multitopology_multiseed",
    "abundant_new": "experiment_2026_09_17_abundant_all_dynamics_multitopology_multiseed_unbounded_lambda1",
}

OUTPUT_DIR = RESULTS_ROOT / "final_math_simplification_comparison_2026_09_17"
PLOT_DIR = OUTPUT_DIR / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_ENVIRONMENTS = 195
EXPECTED_LEARNING_SEEDS = 3
BASELINE_TOL = 1e-9
DELTA_TOL = 1e-12

DYNAMICS_LABELS = {
    "laplacian": "Linear",
    "coca": "COCA",
    "hegselmannkrause": "HK",
    "friedkinjohnsen": "FJ",
    "nonlinearinfluence": "Nonlinear",
    "repulsion": "Repulsion",
}

DYNAMICS_ORDER = [
    "laplacian", "coca", "hegselmannkrause",
    "friedkinjohnsen", "nonlinearinfluence", "repulsion",
]

print("Repository root:", REPO_ROOT)
print("Comparison output:", OUTPUT_DIR)


C:\Users\Chainsword\AppData\Local\Temp\ipykernel_42708\2984534332.py:12: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Repository root: d:\Work\repos\RL\unknown_graph_networks
Comparison output: d:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\results\final_math_simplification_comparison_2026_09_17


## 1. Locate the four benchmark result directories

Automatic discovery searches the normal `experiments/results` location first and then the repository (including archives). If you renamed a result directory, set the corresponding path in `OVERRIDES` above.


In [2]:
PRUNE_DIRS = {".git", ".venv", "venv", "env", "node_modules", "__pycache__", ".idea", ".vscode"}


def find_exact_dir(root: Path, dirname: str) -> list[Path]:
    hits: list[Path] = []
    if not root.exists():
        return hits
    for current, dirs, _files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in PRUNE_DIRS]
        current_path = Path(current)
        if current_path.name == dirname:
            hits.append(current_path)
            dirs[:] = []
    return hits


def locate_result(key: str) -> Path:
    override = OVERRIDES.get(key)
    if override:
        path = Path(override).expanduser()
        if not path.is_absolute():
            path = REPO_ROOT / path
        path = path.resolve()
        if not path.exists():
            raise FileNotFoundError(f"Override for {key} does not exist: {path}")
        return path

    dirname = EXPECTED_DIR_NAMES[key]
    direct = RESULTS_ROOT / dirname
    if direct.exists():
        return direct.resolve()

    hits = find_exact_dir(REPO_ROOT, dirname)
    if not hits:
        raise FileNotFoundError(
            f"Could not locate {key} result directory named {dirname!r}. "
            "If it was renamed in the archive, set OVERRIDES at the top of the notebook."
        )

    # Prefer paths containing 'results', then the shortest path.
    hits.sort(key=lambda p: ("results" not in {part.lower() for part in p.parts}, len(p.parts), str(p)))
    if len(hits) > 1:
        print(f"WARNING: multiple candidates for {key}; using {hits[0]}")
        for h in hits:
            print("  candidate:", h)
    return hits[0].resolve()


RUN_DIRS = {key: locate_result(key) for key in EXPECTED_DIR_NAMES}
for key, path in RUN_DIRS.items():
    print(f"{key:14s}: {path}")


single_old    : D:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\archive\pre_final_2026_09_16\results\experiment_2026_08_17_broad_all_dynamics_multitopology_multiseed
single_new    : D:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\results\experiment_2026_09_17_broad_all_dynamics_multitopology_multiseed_unbounded_lambda1
abundant_old  : D:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\archive\pre_final_2026_09_16\results\experiment_2026_08_24_abundant_all_dynamics_multitopology_multiseed
abundant_new  : D:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\results\experiment_2026_09_17_abundant_all_dynamics_multitopology_multiseed_unbounded_lambda1


## 2. Load and validate the runs

Single-shot results are reconstructed directly from the persistent caches referenced by each `RUN_MANIFEST.json`. Abundant-data results are read from the `combined_summary.csv` created by the final merge pass.


In [3]:
def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


def completion_status(result_dir: Path) -> dict[str, Any]:
    path = result_dir / "COMPUTATION_COMPLETE.json"
    if not path.exists():
        raise FileNotFoundError(
            f"Missing completion marker: {path}\n"
            "The computation/merge pass has not completed successfully."
        )
    payload = read_json(path)
    if payload.get("status") != "success":
        raise RuntimeError(f"Completion marker is not successful: {path}")
    return payload


def locate_study_cache_root(manifest: dict[str, Any]) -> Path:
    recorded = Path(str(manifest.get("cache_root", "")))
    if recorded.exists():
        return recorded

    study_name = str(manifest["study_name"])
    preferred = RESULTS_ROOT / "_trial_cache" / study_name
    if preferred.exists():
        return preferred

    hits = find_exact_dir(REPO_ROOT, study_name)
    hits = [h for h in hits if h.is_dir()]
    if not hits:
        raise FileNotFoundError(
            f"Could not locate cache root for study {study_name!r}. "
            f"Recorded path was {recorded}."
        )
    hits.sort(key=lambda p: ("_trial_cache" not in p.parts, len(p.parts), str(p)))
    return hits[0]


def resolve_single_env_cache(
    env_meta: dict[str, Any],
    manifest: dict[str, Any],
    study_cache_root: Path,
) -> Path:
    recorded = Path(str(env_meta.get("cache_dir", "")))
    if recorded.exists():
        return recorded

    env_id = str(env_meta["environment_id"])
    config_hash = manifest.get("config_hash")
    impl_hash = manifest.get("implementation_hash")
    candidates = list(study_cache_root.glob(f"{env_id}__*"))

    valid: list[Path] = []
    for candidate in candidates:
        mpath = candidate / "baseline_manifest.json"
        if not mpath.exists():
            continue
        try:
            payload = read_json(mpath)
        except Exception:
            continue
        if config_hash is not None and payload.get("config_hash") != config_hash:
            continue
        if impl_hash is not None and payload.get("implementation_hash") != impl_hash:
            continue
        valid.append(candidate)

    if len(valid) != 1:
        raise RuntimeError(
            f"Expected exactly one cache directory for {env_id}; found {len(valid)} "
            f"under {study_cache_root}."
        )
    return valid[0]


def load_single_shot(result_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    completion_status(result_dir)
    manifest_path = result_dir / "RUN_MANIFEST.json"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Missing single-shot manifest: {manifest_path}")
    manifest = read_json(manifest_path)
    study_cache_root = locate_study_cache_root(manifest)

    baseline_rows: list[dict[str, Any]] = []
    learned_rows: list[dict[str, Any]] = []
    learning_seeds = [int(x) for x in manifest["learning_seeds"]]

    for env_meta in manifest["environments"]:
        cache_dir = resolve_single_env_cache(env_meta, manifest, study_cache_root)
        baseline_path = cache_dir / "baseline_summary.json"
        if not baseline_path.exists():
            raise FileNotFoundError(baseline_path)
        baseline_rows.extend(read_json(baseline_path))

        learned_root = cache_dir / "learned"
        for seed in learning_seeds:
            candidates = list(learned_root.glob(f"seed_{seed:02d}_*/summary.json"))
            if len(candidates) != 1:
                raise RuntimeError(
                    f"Expected one learned summary for {env_meta['environment_id']} seed={seed}; "
                    f"found {len(candidates)}"
                )
            learned_rows.append(read_json(candidates[0]))

    baseline = pd.DataFrame(baseline_rows)
    learned = pd.DataFrame(learned_rows)
    return baseline, learned, manifest


def load_abundant(result_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    completion_status(result_dir)
    path = result_dir / "combined_summary.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Re-run the abundant launcher so its final merge pass executes."
        )
    combined = pd.read_csv(path)
    baseline = combined[combined["policy"].isin(["no_control", "uniform", "true_graph"])].copy()
    learned = combined[~combined["policy"].isin(["no_control", "uniform", "true_graph"])].copy()
    return baseline, learned, combined


single_old_base, single_old_learned, single_old_manifest = load_single_shot(RUN_DIRS["single_old"])
single_new_base, single_new_learned, single_new_manifest = load_single_shot(RUN_DIRS["single_new"])
abundant_old_base, abundant_old_learned, abundant_old_combined = load_abundant(RUN_DIRS["abundant_old"])
abundant_new_base, abundant_new_learned, abundant_new_combined = load_abundant(RUN_DIRS["abundant_new"])

print("Loaded all four completed runs.")
print("Single-shot learned rows:", len(single_old_learned), "->", len(single_new_learned))
print("Abundant learned rows:", len(abundant_old_learned), "->", len(abundant_new_learned))


Loaded all four completed runs.
Single-shot learned rows: 585 -> 585
Abundant learned rows: 585 -> 585


In [4]:
def assert_run_shape(name: str, baseline: pd.DataFrame, learned: pd.DataFrame) -> None:
    n_env = learned["environment_id"].nunique()
    if n_env != EXPECTED_ENVIRONMENTS:
        raise RuntimeError(f"{name}: expected {EXPECTED_ENVIRONMENTS} environments, got {n_env}")
    if len(baseline) != EXPECTED_ENVIRONMENTS * 3:
        raise RuntimeError(f"{name}: expected {EXPECTED_ENVIRONMENTS*3} baseline rows, got {len(baseline)}")
    if len(learned) != EXPECTED_ENVIRONMENTS * EXPECTED_LEARNING_SEEDS:
        raise RuntimeError(
            f"{name}: expected {EXPECTED_ENVIRONMENTS*EXPECTED_LEARNING_SEEDS} learned rows, got {len(learned)}"
        )
    if baseline.duplicated(["environment_id", "policy"]).any():
        raise RuntimeError(f"{name}: duplicate baseline rows")
    if learned.duplicated(["environment_id", "learning_seed"]).any():
        raise RuntimeError(f"{name}: duplicate learned seed rows")
    seed_counts = learned.groupby("environment_id").size()
    if not seed_counts.eq(EXPECTED_LEARNING_SEEDS).all():
        raise RuntimeError(f"{name}: not every environment has {EXPECTED_LEARNING_SEEDS} learned seeds")


for name, base, learned in [
    ("single_old", single_old_base, single_old_learned),
    ("single_new", single_new_base, single_new_learned),
    ("abundant_old", abundant_old_base, abundant_old_learned),
    ("abundant_new", abundant_new_base, abundant_new_learned),
]:
    assert_run_shape(name, base, learned)


def compare_environment_sets(old: pd.DataFrame, new: pd.DataFrame, label: str) -> None:
    old_ids = set(old["environment_id"].astype(str))
    new_ids = set(new["environment_id"].astype(str))
    if old_ids != new_ids:
        raise RuntimeError(
            f"{label}: environment sets differ. old-only={sorted(old_ids-new_ids)[:5]}, "
            f"new-only={sorted(new_ids-old_ids)[:5]}"
        )


compare_environment_sets(single_old_learned, single_new_learned, "single-shot")
compare_environment_sets(abundant_old_learned, abundant_new_learned, "abundant")

# Single-shot scientific config should be identical except for the intended
# pipeline-version label and lambda_mix. The alpha-bound removal is an
# implementation/source change, not a benchmark-design change.
old_cfg = dict(single_old_manifest["scientific_config"])
new_cfg = dict(single_new_manifest["scientific_config"])
all_cfg_keys = sorted(set(old_cfg) | set(new_cfg))
config_differences = {
    key: (old_cfg.get(key), new_cfg.get(key))
    for key in all_cfg_keys
    if old_cfg.get(key) != new_cfg.get(key)
}
allowed_config_differences = {"pipeline_version", "lambda_mix"}
unexpected_cfg = sorted(set(config_differences) - allowed_config_differences)
if unexpected_cfg:
    raise RuntimeError(
        "Unexpected single-shot scientific-config differences: " + ", ".join(unexpected_cfg)
    )

old_lambda = float(old_cfg.get("lambda_mix"))
new_lambda = float(new_cfg.get("lambda_mix"))
if not (abs(old_lambda - 0.70) < 1e-12 and abs(new_lambda - 1.0) < 1e-12):
    raise RuntimeError(f"Unexpected lambda comparison: old={old_lambda}, new={new_lambda}")

print("Single-shot config audit passed.")
print("Intentional config differences:", config_differences)
print("Implementation hash changed:", single_old_manifest.get("implementation_hash") != single_new_manifest.get("implementation_hash"))

# Abundant-data design invariants available in the merged outputs.
for col, expected in [
    ("abundant_resets", 100),
    ("abundant_campaigns_per_reset", 5),
    ("train_pairs_total", 2500),
]:
    for label, df in [("old", abundant_old_learned), ("new", abundant_new_learned)]:
        if col not in df.columns:
            raise RuntimeError(f"Abundant {label} results are missing design column {col!r}")
        values = sorted(pd.to_numeric(df[col], errors="coerce").dropna().unique().tolist())
        if values != [expected]:
            raise RuntimeError(f"Abundant {label}: {col}={values}, expected [{expected}]")

print("Abundant-data design invariants passed.")


Single-shot config audit passed.
Intentional config differences: {'lambda_mix': (0.7, 1.0), 'pipeline_version': ('2026-08-17-v1', '2026-09-17-unbounded-lambda1-v1')}
Implementation hash changed: True
Abundant-data design invariants passed.


## 3. Baseline-equivalence audit

Because the simplification only changes the learned identifier/controller model, the fixed `no_control`, `uniform`, and `true_graph` rollouts must remain the same. Any material baseline mismatch would indicate a confound in the rerun.


In [5]:
def baseline_audit(
    regime: str,
    old_df: pd.DataFrame,
    new_df: pd.DataFrame,
    final_col: str,
    path_col: str,
) -> dict[str, Any]:
    cols = ["environment_id", "policy", final_col, path_col]
    old = old_df[cols].rename(columns={final_col: "old_final", path_col: "old_path"})
    new = new_df[cols].rename(columns={final_col: "new_final", path_col: "new_path"})
    paired = old.merge(new, on=["environment_id", "policy"], how="outer", validate="one_to_one", indicator=True)
    if not paired["_merge"].eq("both").all():
        raise RuntimeError(f"{regime}: baseline row identities differ")
    max_final = float(np.max(np.abs(paired["new_final"] - paired["old_final"])))
    max_path = float(np.max(np.abs(paired["new_path"] - paired["old_path"])))
    if max(max_final, max_path) > BASELINE_TOL:
        raise RuntimeError(
            f"{regime}: baseline mismatch exceeds tolerance {BASELINE_TOL}: "
            f"max_final={max_final:.3e}, max_path={max_path:.3e}"
        )
    return {
        "regime": regime,
        "n_baseline_pairs": int(len(paired)),
        "max_abs_final_difference": max_final,
        "max_abs_path_difference": max_path,
        "tolerance": BASELINE_TOL,
        "status": "pass",
    }


audit_rows = [
    baseline_audit("single_shot", single_old_base, single_new_base, "final_mean", "mean_over_campaigns"),
    baseline_audit("abundant", abundant_old_base, abundant_new_base, "mean_end", "mean_over_trajectory"),
]
baseline_audit_df = pd.DataFrame(audit_rows)
baseline_audit_df.to_csv(OUTPUT_DIR / "baseline_equivalence_audit.csv", index=False)
display(baseline_audit_df)
print("Baseline-equivalence audit PASSED.")


,regime,n_baseline_pairs,max_abs_final_difference,max_abs_path_difference,tolerance,status
0,single_shot,585,0.0,0.0,1.000000e-09,pass
1,abundant,585,0.0,0.0,1.000000e-09,pass


Baseline-equivalence audit PASSED.


## 4. Build paired seed-level and environment-level comparisons

Learning seeds are paired within exactly the same environment. For inferential summaries, the three seeds are first averaged inside each environment so that environments—not individual seeds—are the primary comparison units.


In [6]:
def canonical_learned(df: pd.DataFrame, regime: str) -> pd.DataFrame:
    if regime == "single_shot":
        final_col = "final_mean"
        path_col = "mean_over_campaigns"
        diag_map = {
            "model_over_identity": "final_fit_model_over_identity",
            "fit_train_mae": "final_fit_train_mae",
            "fit_identity_mae": "final_fit_identity_mae",
            "fit_n_pairs": "final_fit_n_pairs",
        }
    elif regime == "abundant":
        final_col = "mean_end"
        path_col = "mean_over_trajectory"
        diag_map = {
            "model_over_identity": "model_over_identity",
            "train_mae": "train_mae",
            "identity_mae": "identity_mae",
            "adjacency_mae": "A_MAE",
            "adjacency_frobenius": "A_Fro",
            "centrality_l1": "v_static_L1",
            "fit_elapsed_s": "fit_elapsed_s",
            "fit_steps_run": "fit_steps_run",
            "training_pairs": "train_pairs_total",
        }
    else:
        raise ValueError(regime)

    required = {
        "environment_id", "scenario_class", "family", "dynamics", "learning_seed",
        final_col, path_col,
    }
    missing = sorted(required - set(df.columns))
    if missing:
        raise RuntimeError(f"{regime}: learned results missing columns {missing}")

    out = df[[
        "environment_id", "scenario_class", "family", "dynamics",
        "topology_seed", "opinion_seed", "environment_seed", "learning_seed",
    ]].copy()
    out["final"] = pd.to_numeric(df[final_col], errors="coerce")
    out["path"] = pd.to_numeric(df[path_col], errors="coerce")

    for canonical, source in diag_map.items():
        if source in df.columns:
            out[canonical] = pd.to_numeric(df[source], errors="coerce")

    return out


def canonical_baselines(df: pd.DataFrame, regime: str) -> pd.DataFrame:
    final_col = "final_mean" if regime == "single_shot" else "mean_end"
    path_col = "mean_over_campaigns" if regime == "single_shot" else "mean_over_trajectory"
    out = df[["environment_id", "policy", final_col, path_col]].copy()
    out = out.rename(columns={final_col: "final", path_col: "path"})
    return out


def make_seed_pairs(
    regime: str,
    old_learned: pd.DataFrame,
    new_learned: pd.DataFrame,
    old_baseline: pd.DataFrame,
) -> pd.DataFrame:
    old = canonical_learned(old_learned, regime)
    new = canonical_learned(new_learned, regime)

    keys = ["environment_id", "learning_seed"]
    metadata = ["scenario_class", "family", "dynamics", "topology_seed", "opinion_seed", "environment_seed"]
    metric_cols = [c for c in old.columns if c not in keys + metadata]
    common_metrics = [c for c in metric_cols if c in new.columns]

    old_sel = old[keys + metadata + common_metrics].copy()
    new_sel = new[keys + common_metrics].copy()
    old_sel = old_sel.rename(columns={m: f"old_{m}" for m in common_metrics})
    new_sel = new_sel.rename(columns={m: f"new_{m}" for m in common_metrics})

    paired = old_sel.merge(new_sel, on=keys, how="outer", validate="one_to_one", indicator=True)
    if not paired["_merge"].eq("both").all():
        raise RuntimeError(f"{regime}: learned seed identities differ")
    paired = paired.drop(columns="_merge")

    for metric in common_metrics:
        paired[f"delta_{metric}"] = paired[f"new_{metric}"] - paired[f"old_{metric}"]

    baseline = canonical_baselines(old_baseline, regime)
    uniform = baseline[baseline["policy"] == "uniform"][["environment_id", "final", "path"]].rename(
        columns={"final": "uniform_final", "path": "uniform_path"}
    )
    oracle = baseline[baseline["policy"] == "true_graph"][["environment_id", "final", "path"]].rename(
        columns={"final": "true_graph_final", "path": "true_graph_path"}
    )
    paired = paired.merge(uniform, on="environment_id", how="left", validate="many_to_one")
    paired = paired.merge(oracle, on="environment_id", how="left", validate="many_to_one")

    paired["old_gap_uniform_final"] = paired["old_final"] - paired["uniform_final"]
    paired["new_gap_uniform_final"] = paired["new_final"] - paired["uniform_final"]
    paired["old_gap_true_graph_final"] = paired["old_final"] - paired["true_graph_final"]
    paired["new_gap_true_graph_final"] = paired["new_final"] - paired["true_graph_final"]
    paired.insert(0, "regime", regime)
    return paired


single_seed = make_seed_pairs(
    "single_shot", single_old_learned, single_new_learned, single_old_base
)
abundant_seed = make_seed_pairs(
    "abundant", abundant_old_learned, abundant_new_learned, abundant_old_base
)
seed_pairs = pd.concat([single_seed, abundant_seed], ignore_index=True, sort=False)
seed_pairs.to_csv(OUTPUT_DIR / "paired_seed_level.csv", index=False)

meta_cols = [
    "regime", "environment_id", "scenario_class", "family", "dynamics",
    "topology_seed", "opinion_seed", "environment_seed",
]
value_cols = [c for c in seed_pairs.columns if c not in meta_cols + ["learning_seed"]]
agg_map = {c: "mean" for c in value_cols}
env_pairs = seed_pairs.groupby(meta_cols, dropna=False, as_index=False).agg(agg_map)

# Variability of the formulation change across the three learning seeds.
seed_delta_std = (
    seed_pairs.groupby(meta_cols, dropna=False, as_index=False)["delta_final"]
    .std()
    .rename(columns={"delta_final": "delta_final_across_seed_std"})
)
env_pairs = env_pairs.merge(seed_delta_std, on=meta_cols, how="left", validate="one_to_one")
env_pairs.to_csv(OUTPUT_DIR / "paired_environment_level.csv", index=False)

print("Paired seed rows:", len(seed_pairs))
print("Paired environment rows:", len(env_pairs))
display(env_pairs.head().round(6))


Paired seed rows: 1170
Paired environment rows: 390


,regime,environment_id,scenario_class,family,dynamics,topology_seed,opinion_seed,environment_seed,old_final,old_path,old_model_over_identity,old_fit_train_mae,old_fit_identity_mae,old_fit_n_pairs,new_final,new_path,new_model_over_identity,new_fit_train_mae,new_fit_identity_mae,new_fit_n_pairs,delta_final,delta_path,delta_model_over_identity,delta_fit_train_mae,delta_fit_identity_mae,delta_fit_n_pairs,uniform_final,uniform_path,true_graph_final,true_graph_path,old_gap_uniform_final,new_gap_uniform_final,old_gap_true_graph_final,new_gap_true_graph_final,old_train_mae,old_identity_mae,old_adjacency_mae,old_adjacency_frobenius,old_centrality_l1,old_fit_elapsed_s,old_fit_steps_run,old_training_pairs,new_train_mae,new_identity_mae,new_adjacency_mae,new_adjacency_frobenius,new_centrality_l1,new_fit_elapsed_s,new_fit_steps_run,new_training_pairs,delta_train_mae,delta_identity_mae,delta_adjacency_mae,delta_adjacency_frobenius,delta_centrality_l1,delta_fit_elapsed_s,delta_fit_steps_run,delta_training_pairs,delta_final_across_seed_std
0,abundant,structured__friedkinjohnsen__asymmetric_broadc...,structured_mechanism,asymmetric_broadcast,friedkinjohnsen,NaN,NaN,0.0,0.623709,0.538650,0.929308,NaN,NaN,NaN,0.623709,0.538650,0.305028,NaN,NaN,NaN,0.0,0.0,-0.624280,NaN,NaN,NaN,0.415693,0.383869,0.623709,0.538650,0.208016,0.208016,0.0,0.0,0.035270,0.037953,0.059315,1.495563,0.409970,10.273040,1000.0,2500.0,0.011577,0.037953,0.049535,1.211006,0.301829,9.195661,1000.0,2500.0,-0.023693,0.0,-0.009780,-0.284557,-0.108141,-1.077379,0.0,0.0,0.0
1,abundant,structured__friedkinjohnsen__asymmetric_broadc...,structured_mechanism,asymmetric_broadcast,friedkinjohnsen,NaN,NaN,1.0,0.620043,0.534358,0.929305,NaN,NaN,NaN,0.620043,0.534358,0.311947,NaN,NaN,NaN,0.0,0.0,-0.617358,NaN,NaN,NaN,0.410414,0.378416,0.620043,0.534358,0.209630,0.209630,0.0,0.0,0.033846,0.036420,0.060654,1.529439,0.413457,10.428188,1000.0,2500.0,0.011361,0.036420,0.049379,1.225506,0.293693,9.383792,1000.0,2500.0,-0.022484,0.0,-0.011275,-0.303933,-0.119764,-1.044396,0.0,0.0,0.0
2,abundant,structured__friedkinjohnsen__asymmetric_broadc...,structured_mechanism,asymmetric_broadcast,friedkinjohnsen,NaN,NaN,2.0,0.623326,0.538553,0.928136,NaN,NaN,NaN,0.623326,0.538553,0.308064,NaN,NaN,NaN,0.0,0.0,-0.620071,NaN,NaN,NaN,0.416525,0.384775,0.623326,0.538553,0.206801,0.206801,0.0,0.0,0.033450,0.036040,0.059531,1.508236,0.410539,10.424853,1000.0,2500.0,0.011103,0.036040,0.049084,1.217438,0.301224,9.404486,1000.0,2500.0,-0.022347,0.0,-0.010446,-0.290797,-0.109315,-1.020367,0.0,0.0,0.0
3,abundant,structured__friedkinjohnsen__asymmetric_broadc...,structured_mechanism,asymmetric_broadcast,friedkinjohnsen,NaN,NaN,3.0,0.624522,0.539529,0.929196,NaN,NaN,NaN,0.624522,0.539529,0.317326,NaN,NaN,NaN,0.0,0.0,-0.611869,NaN,NaN,NaN,0.416416,0.384643,0.624522,0.539529,0.208106,0.208106,0.0,0.0,0.032504,0.034981,0.061882,1.567442,0.426963,10.196856,1000.0,2500.0,0.011100,0.034981,0.049738,1.252532,0.298814,9.453282,1000.0,2500.0,-0.021404,0.0,-0.012144,-0.314911,-0.128149,-0.743574,0.0,0.0,0.0
4,abundant,structured__friedkinjohnsen__asymmetric_broadc...,structured_mechanism,asymmetric_broadcast,friedkinjohnsen,NaN,NaN,4.0,0.622267,0.536991,0.929899,NaN,NaN,NaN,0.622267,0.536991,0.309502,NaN,NaN,NaN,0.0,0.0,-0.620397,NaN,NaN,NaN,0.413882,0.381965,0.622267,0.536991,0.208385,0.208385,0.0,0.0,0.033880,0.036434,0.060091,1.523408,0.423562,10.172886,1000.0,2500.0,0.011276,0.036434,0.049151,1.213024,0.309104,9.452824,1000.0,2500.0,-0.022604,0.0,-0.010940,-0.310384,-0.114458,-0.720062,0.0,0.0,0.0


## 5. Paired performance summaries

The 95% confidence intervals below are paired t-intervals over environment-level `new − old` differences. The p-value tests whether the mean paired difference is zero; magnitude and confidence interval are more important than significance alone for judging whether the simplification materially changes the benchmark.


In [7]:
def ci_mean(values: Iterable[float], confidence: float = 0.95) -> tuple[float, float, float]:
    x = np.asarray(list(values), dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    mean = float(np.mean(x))
    if len(x) < 2:
        return mean, np.nan, np.nan
    sem = float(np.std(x, ddof=1) / np.sqrt(len(x)))
    q = float(student_t.ppf(0.5 + confidence / 2.0, df=len(x) - 1))
    return mean, mean - q * sem, mean + q * sem


def performance_summary(group: pd.DataFrame) -> pd.Series:
    delta = pd.to_numeric(group["delta_final"], errors="coerce").to_numpy(float)
    delta = delta[np.isfinite(delta)]
    mean_delta, lo, hi = ci_mean(delta)
    p = float(ttest_1samp(delta, popmean=0.0).pvalue) if len(delta) >= 2 and np.std(delta) > 0 else np.nan
    return pd.Series({
        "n_environments": int(len(group)),
        "old_final_mean": float(group["old_final"].mean()),
        "new_final_mean": float(group["new_final"].mean()),
        "uniform_final_mean": float(group["uniform_final"].mean()),
        "true_graph_final_mean": float(group["true_graph_final"].mean()),
        "new_minus_old_mean": mean_delta,
        "new_minus_old_ci_low": lo,
        "new_minus_old_ci_high": hi,
        "new_minus_old_median": float(np.median(delta)) if len(delta) else np.nan,
        "new_minus_old_std": float(np.std(delta, ddof=1)) if len(delta) >= 2 else np.nan,
        "fraction_improved": float(np.mean(delta > DELTA_TOL)) if len(delta) else np.nan,
        "fraction_worse": float(np.mean(delta < -DELTA_TOL)) if len(delta) else np.nan,
        "fraction_effectively_equal": float(np.mean(np.abs(delta) <= DELTA_TOL)) if len(delta) else np.nan,
        "paired_t_p_value": p,
        "old_gap_uniform_mean": float(group["old_gap_uniform_final"].mean()),
        "new_gap_uniform_mean": float(group["new_gap_uniform_final"].mean()),
        "old_win_rate_vs_uniform": float((group["old_gap_uniform_final"] > 0).mean()),
        "new_win_rate_vs_uniform": float((group["new_gap_uniform_final"] > 0).mean()),
        "old_gap_true_graph_mean": float(group["old_gap_true_graph_final"].mean()),
        "new_gap_true_graph_mean": float(group["new_gap_true_graph_final"].mean()),
        "mean_across_seed_delta_std": float(group["delta_final_across_seed_std"].mean()),
    })


def grouped_summary(df: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
    out = df.groupby(keys, dropna=False, sort=False).apply(performance_summary).reset_index()
    if "dynamics" in out.columns:
        out.insert(out.columns.get_loc("dynamics") + 1, "dynamics_label", out["dynamics"].map(DYNAMICS_LABELS).fillna(out["dynamics"]))
    return out


overall_summary = grouped_summary(env_pairs, ["regime"])
scenario_summary = grouped_summary(env_pairs, ["regime", "scenario_class"])
scenario_dynamics_summary = grouped_summary(env_pairs, ["regime", "scenario_class", "dynamics"])
structured_family_summary = grouped_summary(
    env_pairs[env_pairs["scenario_class"] == "structured_mechanism"],
    ["regime", "dynamics", "family"],
)
unstructured_topology_summary = grouped_summary(
    env_pairs[env_pairs["scenario_class"] == "unstructured_random"],
    ["regime", "dynamics", "topology_seed"],
)
unstructured_opinion_summary = grouped_summary(
    env_pairs[env_pairs["scenario_class"] == "unstructured_random"],
    ["regime", "dynamics", "opinion_seed"],
)

overall_summary.to_csv(OUTPUT_DIR / "summary_overall.csv", index=False)
scenario_summary.to_csv(OUTPUT_DIR / "summary_by_scenario.csv", index=False)
scenario_dynamics_summary.to_csv(OUTPUT_DIR / "summary_by_scenario_dynamics.csv", index=False)
structured_family_summary.to_csv(OUTPUT_DIR / "summary_structured_families.csv", index=False)
unstructured_topology_summary.to_csv(OUTPUT_DIR / "summary_unstructured_topology_seeds.csv", index=False)
unstructured_opinion_summary.to_csv(OUTPUT_DIR / "summary_unstructured_opinion_seeds.csv", index=False)

print("OVERALL")
display(overall_summary.round(6))
print("\nBY SCENARIO + DYNAMICS")
display(scenario_dynamics_summary.round(6))
print("\nSTRUCTURED FAMILIES")
display(structured_family_summary.round(6))


OVERALL


C:\Users\Chainsword\AppData\Local\Temp\ipykernel_42708\524423335.py:45: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  out = df.groupby(keys, dropna=False, sort=False).apply(performance_summary).reset_index()
C:\Users\Chainsword\AppData\Local\Temp\ipykernel_42708\524423335.py:45: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  out = df.groupby(keys, dropna=False, sort=False).apply(performance_summary).reset_in

,regime,n_environments,old_final_mean,new_final_mean,uniform_final_mean,true_graph_final_mean,new_minus_old_mean,new_minus_old_ci_low,new_minus_old_ci_high,new_minus_old_median,new_minus_old_std,fraction_improved,fraction_worse,fraction_effectively_equal,paired_t_p_value,old_gap_uniform_mean,new_gap_uniform_mean,old_win_rate_vs_uniform,new_win_rate_vs_uniform,old_gap_true_graph_mean,new_gap_true_graph_mean,mean_across_seed_delta_std
0,abundant,195.0,0.769767,0.767386,0.640688,0.784534,-0.002381,-0.004948,0.000185,-0.000168,0.018171,0.353846,0.533333,0.112821,0.068785,0.12908,0.126698,0.989744,0.989744,-0.014767,-0.017148,0.010200
1,single_shot,195.0,0.715067,0.763732,0.640688,0.784534,0.048664,0.039911,0.057417,0.042678,0.061974,0.728205,0.271795,0.000000,0.000000,0.07438,0.123044,0.815385,0.984615,-0.069467,-0.020803,0.017788



BY SCENARIO + DYNAMICS


,regime,scenario_class,dynamics,dynamics_label,n_environments,old_final_mean,new_final_mean,uniform_final_mean,true_graph_final_mean,new_minus_old_mean,new_minus_old_ci_low,new_minus_old_ci_high,new_minus_old_median,new_minus_old_std,fraction_improved,fraction_worse,fraction_effectively_equal,paired_t_p_value,old_gap_uniform_mean,new_gap_uniform_mean,old_win_rate_vs_uniform,new_win_rate_vs_uniform,old_gap_true_graph_mean,new_gap_true_graph_mean,mean_across_seed_delta_std
0,abundant,structured_mechanism,friedkinjohnsen,FJ,15.0,0.613681,0.613674,0.417208,0.613842,-0.000007,-0.000022,0.000008,0.000000,0.000027,0.000000,0.066667,0.933333,0.334282,0.196473,0.196466,1.00,1.00,-0.000161,-0.000168,0.000012
1,abundant,structured_mechanism,hegselmannkrause,HK,15.0,0.801743,0.753066,0.669797,0.895423,-0.048678,-0.058089,-0.039267,-0.046592,0.016995,0.000000,1.000000,0.000000,0.000000,0.131947,0.083269,1.00,1.00,-0.093680,-0.142358,0.046790
2,abundant,structured_mechanism,nonlinearinfluence,Nonlinear,15.0,0.865033,0.865087,0.527619,0.865575,0.000053,-0.000077,0.000184,0.000000,0.000236,0.333333,0.200000,0.466667,0.395538,0.337414,0.337468,1.00,1.00,-0.000542,-0.000489,0.000250
3,abundant,unstructured_random,coca,COCA,25.0,0.785436,0.777333,0.736536,0.781852,-0.008103,-0.010041,-0.006164,-0.008293,0.004696,0.000000,1.000000,0.000000,0.000000,0.048900,0.040797,1.00,1.00,0.003584,-0.004519,0.001587
4,abundant,unstructured_random,friedkinjohnsen,FJ,25.0,0.667736,0.685874,0.608784,0.695590,0.018138,0.009962,0.026314,0.010175,0.019807,0.840000,0.160000,0.000000,0.000121,0.058951,0.077089,1.00,1.00,-0.027854,-0.009716,0.019579
5,abundant,unstructured_random,hegselmannkrause,HK,25.0,0.827457,0.833845,0.685553,0.821060,0.006387,0.003406,0.009369,0.004853,0.007223,0.880000,0.120000,0.000000,0.000181,0.141905,0.148292,1.00,1.00,0.006397,0.012785,0.009623
6,abundant,unstructured_random,laplacian,Linear,25.0,0.814504,0.813420,0.693651,0.816446,-0.001084,-0.001459,-0.000708,-0.001023,0.000909,0.120000,0.880000,0.000000,0.000004,0.120853,0.119769,1.00,1.00,-0.001943,-0.003026,0.001088
7,abundant,unstructured_random,nonlinearinfluence,Nonlinear,25.0,0.837183,0.834424,0.693128,0.844726,-0.002758,-0.005417,-0.000100,-0.000574,0.006440,0.280000,0.680000,0.040000,0.042599,0.144055,0.141296,1.00,1.00,-0.007544,-0.010302,0.007081
8,abundant,unstructured_random,repulsion,Repulsion,25.0,0.703596,0.701620,0.610938,0.734787,-0.001976,-0.005890,0.001939,-0.000232,0.009483,0.440000,0.560000,0.000000,0.307936,0.092658,0.090682,0.92,0.92,-0.031191,-0.033167,0.012371
9,single_shot,structured_mechanism,friedkinjohnsen,FJ,15.0,0.541312,0.611277,0.417208,0.613842,0.069966,0.060705,0.079226,0.064000,0.016722,1.000000,0.000000,0.000000,0.000000,0.124104,0.194070,1.00,1.00,-0.072530,-0.002564,0.021163



STRUCTURED FAMILIES


,regime,dynamics,dynamics_label,family,n_environments,old_final_mean,new_final_mean,uniform_final_mean,true_graph_final_mean,new_minus_old_mean,new_minus_old_ci_low,new_minus_old_ci_high,new_minus_old_median,new_minus_old_std,fraction_improved,fraction_worse,fraction_effectively_equal,paired_t_p_value,old_gap_uniform_mean,new_gap_uniform_mean,old_win_rate_vs_uniform,new_win_rate_vs_uniform,old_gap_true_graph_mean,new_gap_true_graph_mean,mean_across_seed_delta_std
0,abundant,friedkinjohnsen,FJ,asymmetric_broadcast,5.0,0.622774,0.622774,0.414586,0.622774,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,1.0,NaN,0.208188,0.208188,1.0,1.0,0.000000,0.000000,0.000000
1,abundant,friedkinjohnsen,FJ,dual_broadcast,5.0,0.609116,0.609095,0.418529,0.609219,-0.000021,-0.000080,0.000037,0.000000,0.000047,0.0,0.2,0.8,0.373901,0.190587,0.190566,1.0,1.0,-0.000103,-0.000124,0.000037
2,abundant,friedkinjohnsen,FJ,split_broadcast,5.0,0.609154,0.609154,0.418509,0.609532,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,1.0,NaN,0.190645,0.190645,1.0,1.0,-0.000378,-0.000378,0.000000
3,abundant,hegselmannkrause,HK,dense_weak_bridge,5.0,0.786859,0.737060,0.670343,0.899413,-0.049800,-0.083321,-0.016278,-0.055024,0.026997,0.0,1.0,0.0,0.014557,0.116516,0.066716,1.0,1.0,-0.112554,-0.162353,0.047438
4,abundant,hegselmannkrause,HK,paired_bridge,5.0,0.805662,0.758965,0.669671,0.896414,-0.046697,-0.059553,-0.033842,-0.046592,0.010354,0.0,1.0,0.0,0.000544,0.135991,0.089294,1.0,1.0,-0.090752,-0.137449,0.048255
5,abundant,hegselmannkrause,HK,ring_bridge,5.0,0.812709,0.763172,0.669376,0.890443,-0.049537,-0.065601,-0.033472,-0.044723,0.012938,0.0,1.0,0.0,0.001022,0.143333,0.093797,1.0,1.0,-0.077734,-0.127271,0.044677
6,abundant,nonlinearinfluence,Nonlinear,asymmetric_broadcast,5.0,0.878131,0.878131,0.525708,0.878131,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,1.0,NaN,0.352423,0.352423,1.0,1.0,0.000000,0.000000,0.000000
7,abundant,nonlinearinfluence,Nonlinear,dual_broadcast,5.0,0.859001,0.858974,0.528638,0.859241,-0.000026,-0.000146,0.000093,0.000000,0.000096,0.2,0.4,0.4,0.575223,0.330362,0.330336,1.0,1.0,-0.000240,-0.000266,0.000112
8,abundant,nonlinearinfluence,Nonlinear,split_broadcast,5.0,0.857969,0.858155,0.528512,0.859355,0.000187,-0.000298,0.000671,0.000356,0.000390,0.8,0.2,0.0,0.345044,0.329457,0.329644,1.0,1.0,-0.001386,-0.001200,0.000640
9,single_shot,friedkinjohnsen,FJ,asymmetric_broadcast,5.0,0.532249,0.619649,0.414586,0.622774,0.087400,0.063251,0.111549,0.099961,0.019449,1.0,0.0,0.0,0.000552,0.117663,0.205063,1.0,1.0,-0.090524,-0.003124,0.058027


## 6. Identification diagnostics

This section compares the diagnostics that are common to each regime before and after the simplification. For MAE/graph-recovery metrics, lower is better; for `model_over_identity`, lower is also better because it is a prediction-error ratio.


In [8]:
DIAGNOSTIC_DIRECTIONS = {
    "model_over_identity": "lower_is_better",
    "fit_train_mae": "lower_is_better",
    "fit_identity_mae": "lower_is_better",
    "fit_n_pairs": "neutral",
    "train_mae": "lower_is_better",
    "identity_mae": "lower_is_better",
    "adjacency_mae": "lower_is_better",
    "adjacency_frobenius": "lower_is_better",
    "centrality_l1": "lower_is_better",
    "fit_elapsed_s": "lower_is_better",
    "fit_steps_run": "lower_is_better",
    "training_pairs": "neutral",
}

rows = []
for (regime, scenario, dynamics), group in env_pairs.groupby(
    ["regime", "scenario_class", "dynamics"], dropna=False, sort=False
):
    for metric, direction in DIAGNOSTIC_DIRECTIONS.items():
        old_col, new_col = f"old_{metric}", f"new_{metric}"
        if old_col not in group.columns or new_col not in group.columns:
            continue
        mask = group[old_col].notna() & group[new_col].notna()
        if not mask.any():
            continue
        old_vals = group.loc[mask, old_col].to_numpy(float)
        new_vals = group.loc[mask, new_col].to_numpy(float)
        delta = new_vals - old_vals
        dmean, dlo, dhi = ci_mean(delta)
        rows.append({
            "regime": regime,
            "scenario_class": scenario,
            "dynamics": dynamics,
            "dynamics_label": DYNAMICS_LABELS.get(dynamics, dynamics),
            "metric": metric,
            "direction": direction,
            "n_environments": int(mask.sum()),
            "old_mean": float(np.mean(old_vals)),
            "new_mean": float(np.mean(new_vals)),
            "new_minus_old_mean": dmean,
            "new_minus_old_ci_low": dlo,
            "new_minus_old_ci_high": dhi,
        })

diagnostic_summary = pd.DataFrame(rows)
diagnostic_summary.to_csv(OUTPUT_DIR / "identification_diagnostics_before_after.csv", index=False)
display(diagnostic_summary.round(6))


,regime,scenario_class,dynamics,dynamics_label,metric,direction,n_environments,old_mean,new_mean,new_minus_old_mean,new_minus_old_ci_low,new_minus_old_ci_high
0,abundant,structured_mechanism,friedkinjohnsen,FJ,model_over_identity,lower_is_better,15,0.926232,0.303143,-0.623089,-0.629343,-0.616834
1,abundant,structured_mechanism,friedkinjohnsen,FJ,train_mae,lower_is_better,15,0.033840,0.011112,-0.022728,-0.024292,-0.021164
2,abundant,structured_mechanism,friedkinjohnsen,FJ,identity_mae,lower_is_better,15,0.036543,0.036543,0.000000,0.000000,0.000000
3,abundant,structured_mechanism,friedkinjohnsen,FJ,adjacency_mae,lower_is_better,15,0.065089,0.050849,-0.014241,-0.016220,-0.012261
4,abundant,structured_mechanism,friedkinjohnsen,FJ,adjacency_frobenius,lower_is_better,15,1.586934,1.232564,-0.354369,-0.397101,-0.311638
5,abundant,structured_mechanism,friedkinjohnsen,FJ,centrality_l1,lower_is_better,15,0.440048,0.297653,-0.142395,-0.157668,-0.127122
6,abundant,structured_mechanism,friedkinjohnsen,FJ,fit_elapsed_s,lower_is_better,15,10.211920,9.445273,-0.766647,-0.855014,-0.678280
7,abundant,structured_mechanism,friedkinjohnsen,FJ,fit_steps_run,lower_is_better,15,1000.000000,1000.000000,0.000000,0.000000,0.000000
8,abundant,structured_mechanism,friedkinjohnsen,FJ,training_pairs,neutral,15,2500.000000,2500.000000,0.000000,0.000000,0.000000
9,abundant,structured_mechanism,hegselmannkrause,HK,model_over_identity,lower_is_better,15,1.085269,1.146201,0.060932,0.051220,0.070645


## 7. Comparison figures

Figures are saved under `results/final_math_simplification_comparison_2026_09_17/plots/`.


In [9]:
mpl.rcdefaults()
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})


def savefig(fig: plt.Figure, stem: str) -> None:
    fig.savefig(PLOT_DIR / f"{stem}.png")
    fig.savefig(PLOT_DIR / f"{stem}.pdf")
    plt.close(fig)

# 1) Distribution of environment-level final-performance change by regime.
fig, ax = plt.subplots(figsize=(6.4, 3.8))
data = [
    env_pairs.loc[env_pairs["regime"] == regime, "delta_final"].to_numpy(float)
    for regime in ["single_shot", "abundant"]
]
ax.boxplot(data, showmeans=True)
ax.set_xticks([1, 2])
ax.set_xticklabels(["Single shot", "Abundant"])
ax.axhline(0.0, linewidth=1.0, linestyle="--")
ax.set_ylabel("Final mean: new − old")
ax.set_title("Effect of λ=1 + unbounded α")
savefig(fig, "01_environment_delta_distribution")

# 2/3) Mean paired delta by scenario+dynamics, one figure per regime.
for regime in ["single_shot", "abundant"]:
    sub = scenario_dynamics_summary[scenario_dynamics_summary["regime"] == regime].copy()
    scenario_rank = {"unstructured_random": 0, "structured_mechanism": 1}
    dyn_rank = {d: i for i, d in enumerate(DYNAMICS_ORDER)}
    sub["_s"] = sub["scenario_class"].map(scenario_rank).fillna(99)
    sub["_d"] = sub["dynamics"].map(dyn_rank).fillna(99)
    sub = sub.sort_values(["_s", "_d"])
    labels = [
        ("generic" if s == "unstructured_random" else "structured") + " / " + DYNAMICS_LABELS.get(d, d)
        for s, d in zip(sub["scenario_class"], sub["dynamics"])
    ]
    y = sub["new_minus_old_mean"].to_numpy(float)
    lo = y - sub["new_minus_old_ci_low"].to_numpy(float)
    hi = sub["new_minus_old_ci_high"].to_numpy(float) - y
    x = np.arange(len(sub))
    fig, ax = plt.subplots(figsize=(8.0, 4.2))
    ax.errorbar(x, y, yerr=np.vstack([lo, hi]), fmt="o", capsize=3)
    ax.axhline(0.0, linewidth=1.0, linestyle="--")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=35, ha="right")
    ax.set_ylabel("Final mean: new − old")
    ax.set_title(("Single shot" if regime == "single_shot" else "Abundant data") + ": paired change by dynamics")
    fig.tight_layout()
    savefig(fig, f"02_{regime}_delta_by_dynamics")

# 4/5) Old vs new environment-level performance.
for regime in ["single_shot", "abundant"]:
    sub = env_pairs[env_pairs["regime"] == regime]
    x = sub["old_final"].to_numpy(float)
    y = sub["new_final"].to_numpy(float)
    low = float(min(x.min(), y.min()))
    high = float(max(x.max(), y.max()))
    fig, ax = plt.subplots(figsize=(5.0, 5.0))
    for scenario, group in sub.groupby("scenario_class"):
        label = "generic" if scenario == "unstructured_random" else "structured"
        ax.scatter(group["old_final"], group["new_final"], s=18, alpha=0.7, label=label)
    ax.plot([low, high], [low, high], linestyle="--", linewidth=1.0)
    ax.set_xlabel("Old final mean")
    ax.set_ylabel("Simplified final mean")
    ax.set_title(("Single shot" if regime == "single_shot" else "Abundant data") + ": environment-level before/after")
    ax.legend(frameon=False)
    savefig(fig, f"03_{regime}_old_vs_new")

# 6) Structured-family change, with regime encoded by marker/offset.
sub = structured_family_summary.copy()
sub["label"] = sub["dynamics"].map(DYNAMICS_LABELS).fillna(sub["dynamics"]) + " / " + sub["family"].astype(str)
labels = list(dict.fromkeys(sub["label"].tolist()))
fig, ax = plt.subplots(figsize=(9.0, 4.8))
for offset, regime, marker in [(-0.12, "single_shot", "o"), (0.12, "abundant", "s")]:
    r = sub[sub["regime"] == regime].set_index("label").reindex(labels)
    y = r["new_minus_old_mean"].to_numpy(float)
    lo = y - r["new_minus_old_ci_low"].to_numpy(float)
    hi = r["new_minus_old_ci_high"].to_numpy(float) - y
    x = np.arange(len(labels), dtype=float) + offset
    ax.errorbar(x, y, yerr=np.vstack([lo, hi]), fmt=marker, capsize=3, label=regime.replace("_", " "))
ax.axhline(0.0, linewidth=1.0, linestyle="--")
ax.set_xticks(np.arange(len(labels)))
ax.set_xticklabels(labels, rotation=38, ha="right")
ax.set_ylabel("Final mean: new − old")
ax.set_title("Structured families: effect of simplified formulation")
ax.legend(frameon=False)
fig.tight_layout()
savefig(fig, "04_structured_family_delta")

print("Saved figures to:", PLOT_DIR)


Saved figures to: d:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\results\final_math_simplification_comparison_2026_09_17\plots


## 8. Decision-focused report

This final cell writes a concise Markdown report and highlights the structured HK result explicitly, because that was the main caveat seen in the ablation study.


In [10]:
def fmt(x: float, digits: int = 6) -> str:
    return "nan" if not np.isfinite(x) else f"{x:.{digits}f}"


def table_markdown(df: pd.DataFrame) -> str:
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_csv(index=False)

key_cols = [
    "regime", "n_environments", "old_final_mean", "new_final_mean", "uniform_final_mean", "true_graph_final_mean",
    "new_minus_old_mean", "new_minus_old_ci_low", "new_minus_old_ci_high",
    "fraction_improved", "fraction_worse", "old_win_rate_vs_uniform", "new_win_rate_vs_uniform",
]

hk_rows = scenario_dynamics_summary[
    (scenario_dynamics_summary["scenario_class"] == "structured_mechanism")
    & (scenario_dynamics_summary["dynamics"] == "hegselmannkrause")
].copy()

ranked = scenario_dynamics_summary.copy()
ranked = ranked.sort_values("new_minus_old_mean")
worst = ranked.head(5)[[
    "regime", "scenario_class", "dynamics_label", "new_minus_old_mean",
    "new_minus_old_ci_low", "new_minus_old_ci_high",
]]
best = ranked.tail(5).sort_values("new_minus_old_mean", ascending=False)[[
    "regime", "scenario_class", "dynamics_label", "new_minus_old_mean",
    "new_minus_old_ci_low", "new_minus_old_ci_high",
]]

report_lines = [
    "# Final before/after comparison: math simplification",
    "",
    "Compared the previous bounded-α, λ=0.70 benchmark against the final λ=1, unbounded-α formulation using paired environments and learning seeds.",
    "",
    "## Validation",
    "",
    f"- Single-shot environments: {single_old_learned['environment_id'].nunique()} (paired exactly).",
    f"- Abundant environments: {abundant_old_learned['environment_id'].nunique()} (paired exactly).",
    f"- Learning seeds per environment: {EXPECTED_LEARNING_SEEDS}.",
    f"- Baseline tolerance: {BASELINE_TOL:g}; baseline audit passed for both regimes.",
    f"- Single-shot λ: {old_lambda:g} → {new_lambda:g}.",
    f"- Unexpected single-shot scientific-config differences: none.",
    "",
    "## Overall paired performance",
    "",
    table_markdown(overall_summary[key_cols].round(6)),
    "",
    "## Structured HK (the ablation caveat)",
    "",
    table_markdown(hk_rows[key_cols + ["paired_t_p_value"]].round(6)),
    "",
    "## Largest negative scenario/dynamics changes",
    "",
    table_markdown(worst.round(6)),
    "",
    "## Largest positive scenario/dynamics changes",
    "",
    table_markdown(best.round(6)),
    "",
    "## Interpretation note",
    "",
    "Positive `new_minus_old` means the simplified formulation produced a higher final mean opinion. "
    "The confidence intervals are paired across environments after averaging the three learning seeds within each environment. "
    "Use effect magnitude and consistency together with the confidence interval; a non-significant p-value alone is not evidence of equivalence.",
]

report_path = OUTPUT_DIR / "FINAL_COMPARISON_REPORT.md"
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

print("\nFINAL OVERALL COMPARISON")
display(overall_summary[key_cols].round(6))
print("\nSTRUCTURED HK CHECK")
display(hk_rows[key_cols + ["paired_t_p_value"]].round(6))
print("\nMost negative scenario/dynamics changes")
display(worst.round(6))
print("\nMost positive scenario/dynamics changes")
display(best.round(6))
print("\nReport written to:", report_path)
print("All comparison outputs:", OUTPUT_DIR)



FINAL OVERALL COMPARISON


,regime,n_environments,old_final_mean,new_final_mean,uniform_final_mean,true_graph_final_mean,new_minus_old_mean,new_minus_old_ci_low,new_minus_old_ci_high,fraction_improved,fraction_worse,old_win_rate_vs_uniform,new_win_rate_vs_uniform
0,abundant,195.0,0.769767,0.767386,0.640688,0.784534,-0.002381,-0.004948,0.000185,0.353846,0.533333,0.989744,0.989744
1,single_shot,195.0,0.715067,0.763732,0.640688,0.784534,0.048664,0.039911,0.057417,0.728205,0.271795,0.815385,0.984615



STRUCTURED HK CHECK


,regime,n_environments,old_final_mean,new_final_mean,uniform_final_mean,true_graph_final_mean,new_minus_old_mean,new_minus_old_ci_low,new_minus_old_ci_high,fraction_improved,fraction_worse,old_win_rate_vs_uniform,new_win_rate_vs_uniform,paired_t_p_value
1,abundant,15.0,0.801743,0.753066,0.669797,0.895423,-0.048678,-0.058089,-0.039267,0.0,1.0,1.0,1.0,0.000000
10,single_shot,15.0,0.813495,0.876433,0.669797,0.895423,0.062938,0.046821,0.079055,1.0,0.0,1.0,1.0,0.000001



Most negative scenario/dynamics changes


,regime,scenario_class,dynamics_label,new_minus_old_mean,new_minus_old_ci_low,new_minus_old_ci_high
1,abundant,structured_mechanism,HK,-0.048678,-0.058089,-0.039267
15,single_shot,unstructured_random,Linear,-0.012549,-0.020424,-0.004674
3,abundant,unstructured_random,COCA,-0.008103,-0.010041,-0.006164
12,single_shot,unstructured_random,COCA,-0.006158,-0.011410,-0.000906
7,abundant,unstructured_random,Nonlinear,-0.002758,-0.005417,-0.000100



Most positive scenario/dynamics changes


,regime,scenario_class,dynamics_label,new_minus_old_mean,new_minus_old_ci_low,new_minus_old_ci_high
16,single_shot,unstructured_random,Nonlinear,0.109270,0.083048,0.135492
14,single_shot,unstructured_random,HK,0.096082,0.064105,0.128060
11,single_shot,structured_mechanism,Nonlinear,0.072676,0.063717,0.081634
9,single_shot,structured_mechanism,FJ,0.069966,0.060705,0.079226
13,single_shot,unstructured_random,FJ,0.069891,0.052076,0.087707



Report written to: d:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\results\final_math_simplification_comparison_2026_09_17\FINAL_COMPARISON_REPORT.md
All comparison outputs: d:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\results\final_math_simplification_comparison_2026_09_17
